In [41]:
import duckdb 
from pathlib import Path

In [42]:
con = duckdb.connect()
SPY_PATH = 'C:/Users/ignac/Proyectos/inflation-hedge-analysis/data/raw/prices_SPY.csv'

In [43]:
duckdb.sql("SELECT * FROM read_csv('C:/Users/ignac/Proyectos/inflation-hedge-analysis/data/raw/prices_SPY.csv') LIMIT 5")

DateDATE,Adj CloseDOUBLE,CloseDOUBLE,HighDOUBLE,LowDOUBLE,OpenDOUBLE,VolumeBIGINT
2005-01-03,80.9735336303711,120.30000305175781,121.76000213623047,119.9000015258789,121.55999755859375,55748000
2005-01-04,79.98406219482422,118.83000183105469,120.54000091552734,118.44000244140625,120.45999908447266,69167600
2005-01-05,79.4321517944336,118.01000213623047,119.25,118.0,118.73999786376953,65667300
2005-01-06,79.83599090576172,118.61000061035156,119.1500015258789,118.26000213623047,118.44000244140625,47814700
2005-01-07,79.7215576171875,118.44000244140625,119.2300033569336,118.12999725341797,118.97000122070312,55847700


In [44]:
duckdb.sql(f"DESCRIBE SELECT * FROM read_csv('{SPY_PATH}')") 

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
Date,DATE,YES,NULL,NULL,NULL
Adj Close,DOUBLE,YES,NULL,NULL,NULL
Close,DOUBLE,YES,NULL,NULL,NULL
High,DOUBLE,YES,NULL,NULL,NULL
Low,DOUBLE,YES,NULL,NULL,NULL
Open,DOUBLE,YES,NULL,NULL,NULL
Volume,BIGINT,YES,NULL,NULL,NULL


In [45]:
DBC_PATH = 'C:/Users/ignac/Proyectos/inflation-hedge-analysis/data/raw/prices_DBC.csv'

duckdb.sql(f"SELECT MAX(DATE) FROM read_csv('{DBC_PATH}')")

max(DATE)DATE
2026-10-23


In [46]:
duckdb.sql(f"SELECT MIN(DATE) FROM read_csv('{DBC_PATH}')")

min(DATE)DATE
2006-02-06


In [47]:
duckdb.sql(f"SELECT COUNT(*) FROM read_csv('{DBC_PATH}')")

count_star()BIGINT
5212


In [48]:
duckdb.sql(f"""
    SELECT COUNT(*) - COUNT("Adj Close") AS null_adj_close
    FROM read_csv('{DBC_PATH}')
""")

null_adj_closeBIGINT
0


In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

cpi_path = RAW_DIR / "fred_CPIAUCSL.json"

c:\Users\ignac\Proyectos\inflation-hedge-analysis\notebooks


In [57]:
duckdb.sql(f"DESCRIBE SELECT * FROM read_json('{cpi_path.as_posix()}')").df()

,column_name,column_type,null,key,default,extra
0,realtime_start,DATE,YES,None,None,None
1,realtime_end,DATE,YES,None,None,None
2,observation_start,DATE,YES,None,None,None
3,observation_end,DATE,YES,None,None,None
4,units,VARCHAR,YES,None,None,None
5,output_type,BIGINT,YES,None,None,None
6,file_type,VARCHAR,YES,None,None,None
7,order_by,VARCHAR,YES,None,None,None
8,sort_order,VARCHAR,YES,None,None,None
9,count,BIGINT,YES,None,None,None


In [65]:
duckdb.sql(f" SELECT UNNEST(observations, recursive := true) FROM read_json('{cpi_path.as_posix()}')")

realtime_startDATE,realtime_endDATE,dateDATE,valueVARCHAR
2026-10-22,2026-10-22,2005-01-01,191.600
2026-10-22,2026-10-22,2005-02-01,192.400
2026-10-22,2026-10-22,2005-03-01,193.100
2026-10-22,2026-10-22,2005-04-01,193.700
2026-10-22,2026-10-22,2005-05-01,193.600
2026-10-22,2026-10-22,2005-06-01,193.700
2026-10-22,2026-10-22,2005-07-01,194.900
2026-10-22,2026-10-22,2005-08-01,196.100
2026-10-22,2026-10-22,2005-09-01,198.800
2026-10-22,2026-10-22,2005-10-01,199.100


In [66]:
duckdb.sql(f"DESCRIBE SELECT UNNEST(observations, recursive := true) FROM read_json('{cpi_path.as_posix()}')")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
realtime_start,DATE,YES,NULL,NULL,NULL
realtime_end,DATE,YES,NULL,NULL,NULL
date,DATE,YES,NULL,NULL,NULL
value,VARCHAR,YES,NULL,NULL,NULL


In [69]:
duckdb.sql(f"""
    SELECT
        obs.date  AS date,
        TRY_CAST(obs.value AS DOUBLE) AS value
    FROM (
        SELECT UNNEST(observations) AS obs
        FROM read_json('{cpi_path.as_posix()}')
    ) AS t
""")

dateDATE,valueDOUBLE
2005-01-01,191.6
2005-02-01,192.4
2005-03-01,193.1
2005-04-01,193.7
2005-05-01,193.6
2005-06-01,193.7
2005-07-01,194.9
2005-08-01,196.1
2005-09-01,198.8
2005-10-01,199.1


In [ ]:
duckdb.sql(f"""
    SELECT
        COUNT(*)
        COUNT(TRY_CAST(obs.value AS DOUBLE)) AS value
    FROM (
        SELECT UNNEST(observations) AS obs
        FROM read_json('{cpi_path.as_posix()}')
    ) AS t
""")

ParserException: Parser Error: syntax error at or near "("

LINE 4:         COUNT(TRY_CAST(obs.value AS DOUBLE)) AS value
                     ^

In [ ]:
# Casting value to be DOBLE in CPI 

duckdb.sql(f"""
    SELECT
        obs.date, CAST(NULLIF(obs.value, '.') AS DOUBLE) AS value
    FROM (
        SELECT UNNEST(observations) AS obs
        FROM read_json('{cpi_path.as_posix()}')
    ) AS t
""")

dateDATE,valueDOUBLE
2005-01-01,191.6
2005-02-01,192.4
2005-03-01,193.1
2005-04-01,193.7
2005-05-01,193.6
2005-06-01,193.7
2005-07-01,194.9
2005-08-01,196.1
2005-09-01,198.8
2005-10-01,199.1


In [ ]:
#Extracting series_id from FILENAME in FRED FILES

duckdb.sql(rf"""
    SELECT
        regexp_extract(filename, 'fred_(.+)\.json', 1) AS series_id,
        obs.date                                       AS date,
        CAST(NULLIF(obs.value, '.') AS DOUBLE)         AS value
    FROM (
        SELECT filename, UNNEST(observations) AS obs
        FROM read_json('{RAW_DIR.as_posix()}/fred_*.json', filename = true)
    ) AS t
""")

series_idVARCHAR,dateDATE,valueDOUBLE
CPIAUCSL,2005-01-01,191.6
CPIAUCSL,2005-02-01,192.4
CPIAUCSL,2005-03-01,193.1
CPIAUCSL,2005-04-01,193.7
CPIAUCSL,2005-05-01,193.6
CPIAUCSL,2005-06-01,193.7
CPIAUCSL,2005-07-01,194.9
CPIAUCSL,2005-08-01,196.1
CPIAUCSL,2005-09-01,198.8
CPIAUCSL,2005-10-01,199.1


In [ ]:
# Counting how many rows each series has

duckdb.sql(rf"""
    SELECT
        regexp_extract(filename, 'fred_(.+)\.json', 1) AS series_id,
        COUNT(*)
    FROM (
        SELECT filename, UNNEST(observations) AS obs
        FROM read_json('{RAW_DIR.as_posix()}/fred_*.json', filename = true)
    ) AS t
    GROUP BY series_id
""")

series_idVARCHAR,count_star()BIGINT
TB3MS,261
CPIAUCSL,261


In [ ]:
#Extracting ticker_id from all the csvs

duckdb.sql(rf"""
    SELECT
        Date as date,
        regexp_extract(filename, 'prices_(.+)\.csv', 1) AS ticker,
        "Adj Close" AS adj_close,
        Close       AS close,
        High        AS high,
        Low         AS low,
        Open        AS open,
        Volume      AS volume
    FROM read_csv('{RAW_DIR.as_posix()}/prices_*.csv', filename = true)
""")

dateDATE,tickerVARCHAR,adj_closeDOUBLE,closeDOUBLE,highDOUBLE,lowDOUBLE,openDOUBLE,volumeBIGINT
2006-02-06,DBC,19.185100555419922,24.200000762939453,24.65999984741211,24.100000381469727,24.549999237060547,771500
2006-02-07,DBC,18.630157470703125,23.5,24.200000762939453,23.450000762939453,24.049999237060547,674400
2006-02-08,DBC,18.550880432128906,23.399999618530273,23.670000076293945,23.329999923706055,23.489999771118164,335400
2006-02-09,DBC,18.72528839111328,23.6200008392334,23.81999969482422,23.510000228881836,23.559999465942383,705900
2006-02-10,DBC,18.3843994140625,23.190000534057617,23.690000534057617,23.100000381469727,23.649999618530273,276700
2006-02-13,DBC,18.0989990234375,22.829999923706055,23.200000762939453,22.770000457763672,23.190000534057617,237600
2006-02-14,DBC,17.948373794555664,22.639999389648438,22.829999923706055,22.610000610351562,22.799999237060547,256400
2006-02-15,DBC,17.686756134033203,22.309999465942383,22.729999542236328,22.239999771118164,22.700000762939453,404800
2006-02-16,DBC,17.900808334350586,22.579999923706055,22.6200008392334,22.25,22.329999923706055,463600
2006-02-17,DBC,18.15449333190918,22.899999618530273,22.920000076293945,22.68000030517578,22.719999313354492,467700


In [ ]:
#Total Rows

duckdb.sql(rf"""
    SELECT
        COUNT(regexp_extract(filename, 'prices_(.+)\.csv', 1)) AS TICKER
    FROM read_csv('{RAW_DIR.as_posix()}/prices_*.csv', filename = true)
""")

TICKERBIGINT
32647


In [ ]:
# How many rows per ticker

duckdb.sql(rf"""
    SELECT
        COUNT(*),
        regexp_extract(filename, 'prices_(.+)\.csv', 1) AS ticker
    FROM read_csv('{RAW_DIR.as_posix()}/prices_*.csv', filename = true)
    GROUP BY ticker
""")

count_star()BIGINT,tickerVARCHAR
5487,SPY
5487,TIP
5487,TLT
5487,GLD
5212,DBC
5487,VNQ


In [96]:
#View that will be utiliced for checking nulls

duckdb.sql(rf"""
    CREATE OR REPLACE VIEW stg_prices_preview AS
    SELECT
        Date                                            AS date,
        regexp_extract(filename, 'prices_(.+)\.csv', 1) AS ticker,
        "Adj Close"                                     AS adj_close,
        Close                                           AS close,
        High                                            AS high,
        Low                                             AS low,
        Open                                            AS open,
        Volume                                          AS volume
    FROM read_csv('{RAW_DIR.as_posix()}/prices_*.csv', filename = true)
""")

CountBIGINT


In [97]:
duckdb.sql("SELECT COUNT(*) AS total_rows FROM stg_prices_preview")

total_rowsBIGINT
32647


In [98]:
duckdb.sql("""
    SELECT
        ticker,
        COUNT(*),
        MIN(DATE) as first_date, 
        MAX(DATE) as last_date,
    FROM stg_prices_preview
    GROUP BY ticker    
    ORDER BY ticker
""")

tickerVARCHAR,count_star()BIGINT,first_dateDATE,last_dateDATE
DBC,5212,2006-02-06,2026-10-23
GLD,5487,2005-01-03,2026-10-23
SPY,5487,2005-01-03,2026-10-23
TIP,5487,2005-01-03,2026-10-23
TLT,5487,2005-01-03,2026-10-23
VNQ,5487,2005-01-03,2026-10-23


In [100]:
duckdb.sql("""
    SELECT 
        date,
        ticker,
        count(*) as n
    FROM stg_prices_preview
    GROUP BY date, ticker 
    HAVING COUNT(*) > 1 
""")

dateDATE,tickerVARCHAR,nBIGINT
